# Feature Engineering
## Telco Customer Churn Dataset

This notebook handles data cleaning and feature engineering for the churn prediction model.

### Contents:
1. Data Loading & Cleaning
2. Feature Creation
3. Feature Encoding
4. Feature Scaling
5. Data Splitting
6. Save Processed Data

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings

# Add src to path
sys.path.append('../src')

from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
import joblib

warnings.filterwarnings('ignore')

# Import project modules
from config import (
    RAW_DATA_DIR, PROCESSED_DATA_DIR, TRAIN_DATA_DIR, VAL_DATA_DIR, TEST_DATA_DIR,
    MODELS_DIR, CATEGORICAL_COLUMNS, NUMERICAL_COLUMNS, TARGET_COLUMN,
    RANDOM_STATE, TEST_SIZE, VAL_SIZE, create_directories
)

create_directories()
print('Libraries imported and directories created!')

## 1. Data Loading & Cleaning

In [ ]:
# Load raw data
df = pd.read_csv(RAW_DATA_DIR / 'WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'Original shape: {df.shape}')
df.head()

In [ ]:
# Clean TotalCharges - convert to numeric
print('Before cleaning TotalCharges:')
print(f'Data type: {df["TotalCharges"].dtype}')
print(f'Non-numeric values: {pd.to_numeric(df["TotalCharges"], errors="coerce").isnull().sum()}')

# Convert and fill missing with MonthlyCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['MonthlyCharges'], inplace=True)

print('\nAfter cleaning TotalCharges:')
print(f'Data type: {df["TotalCharges"].dtype}')
print(f'Missing values: {df["TotalCharges"].isnull().sum()}')

In [ ]:
# Convert SeniorCitizen to categorical
print('Before conversion:')
print(df['SeniorCitizen'].value_counts())

df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

print('\nAfter conversion:')
print(df['SeniorCitizen'].value_counts())

In [ ]:
# Create binary target
df['Churn_Binary'] = df['Churn'].map({'No': 0, 'Yes': 1})
print('Target distribution:')
print(df['Churn_Binary'].value_counts())

In [ ]:
# Remove duplicates
initial_len = len(df)
df = df.drop_duplicates()
print(f'Removed {initial_len - len(df)} duplicate rows')
print(f'Final shape after cleaning: {df.shape}')

## 2. Feature Creation

In [ ]:
# Create tenure groups
bins = [0, 12, 24, 48, 72]
labels = ['0-12 months', '12-24 months', '24-48 months', '48+ months']
df['TenureGroup'] = pd.cut(df['tenure'], bins=bins, labels=labels, include_lowest=True)

print('Tenure Group distribution:')
print(df['TenureGroup'].value_counts().sort_index())

In [ ]:
# Create charge-related features

# Average monthly charge (total / tenure)
df['AvgMonthlyCharge'] = np.where(
    df['tenure'] > 0,
    df['TotalCharges'] / df['tenure'],
    df['MonthlyCharges']
)

# Charge increase over time
df['ChargeIncrease'] = df['MonthlyCharges'] - df['AvgMonthlyCharge']

# High value customer flag
median_charges = df['MonthlyCharges'].median()
df['HighValue'] = (df['MonthlyCharges'] > median_charges).astype(int)

print(f'Median Monthly Charges: ${median_charges:.2f}')
print(f'High Value Customers: {df["HighValue"].sum()} ({df["HighValue"].mean()*100:.1f}%)')

In [ ]:
# Create service count feature
service_columns = [
    'PhoneService', 'MultipleLines', 'InternetService', 
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]

# Count services (Yes = 1, others = 0)
df['TotalServices'] = sum((df[col] == 'Yes').astype(int) for col in service_columns)

print('Total Services distribution:')
print(df['TotalServices'].value_counts().sort_index())

In [ ]:
# Create contract-related features

# Short-term contract flag
df['ShortTermContract'] = (df['Contract'] == 'Month-to-month').astype(int)

# Automatic payment flag
df['AutomaticPayment'] = df['PaymentMethod'].apply(
    lambda x: 1 if 'automatic' in str(x).lower() else 0
)

print(f'Short-term contracts: {df["ShortTermContract"].mean()*100:.1f}%')
print(f'Automatic payments: {df["AutomaticPayment"].mean()*100:.1f}%')

In [ ]:
# View new features
new_features = ['TenureGroup', 'AvgMonthlyCharge', 'ChargeIncrease', 
                'HighValue', 'TotalServices', 'ShortTermContract', 'AutomaticPayment']

print('New Features Summary:')
df[new_features + ['Churn_Binary']].describe()

## 3. Feature Encoding

In [ ]:
# Define categorical columns to encode
cat_columns = CATEGORICAL_COLUMNS.copy()
cat_columns.append('SeniorCitizen')  # Added after conversion

print('Categorical columns to encode:')
print(cat_columns)

In [ ]:
# Label encode categorical columns
encoders = {}

for col in cat_columns:
    le = LabelEncoder()
    df[f'{col}_encoded'] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
    print(f'{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

In [ ]:
# View encoded columns
encoded_cols = [f'{col}_encoded' for col in cat_columns]
df[encoded_cols].head()

## 4. Feature Scaling

In [ ]:
# Define numerical columns to scale
numerical_cols = NUMERICAL_COLUMNS.copy()
numerical_cols.extend(['AvgMonthlyCharge', 'ChargeIncrease', 'TotalServices'])

print('Numerical columns to scale:')
print(numerical_cols)

In [ ]:
# Check distributions before scaling
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    axes[i].hist(df[col], bins=30, edgecolor='black', alpha=0.7)
    axes[i].set_title(f'{col} Distribution')
    axes[i].set_xlabel(col)

plt.tight_layout()
plt.show()

In [ ]:
# Apply StandardScaler
scaler = StandardScaler()
scaled_values = scaler.fit_transform(df[numerical_cols])

# Create scaled column names
scaled_cols = [f'{col}_scaled' for col in numerical_cols]
df[scaled_cols] = scaled_values

print('Scaled features summary:')
df[scaled_cols].describe().round(2)

In [ ]:
# Check distributions after scaling
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(scaled_cols):
    axes[i].hist(df[col], bins=30, edgecolor='black', alpha=0.7, color='green')
    axes[i].set_title(f'{col}')
    axes[i].axvline(0, color='red', linestyle='--')

plt.tight_layout()
plt.show()

## 5. Data Splitting

In [ ]:
# Split data into train, validation, and test sets
print(f'Total samples: {len(df)}')
print(f'Test size: {TEST_SIZE}')
print(f'Validation size: {VAL_SIZE}')
print(f'Random state: {RANDOM_STATE}')

In [ ]:
# First split: train+val vs test
train_val, test = train_test_split(
    df, 
    test_size=TEST_SIZE, 
    random_state=RANDOM_STATE,
    stratify=df['Churn_Binary']
)

# Second split: train vs val
val_size_adjusted = VAL_SIZE / (1 - TEST_SIZE)
train, val = train_test_split(
    train_val,
    test_size=val_size_adjusted,
    random_state=RANDOM_STATE,
    stratify=train_val['Churn_Binary']
)

print(f'Train set: {len(train)} samples ({len(train)/len(df)*100:.1f}%)')
print(f'Validation set: {len(val)} samples ({len(val)/len(df)*100:.1f}%)')
print(f'Test set: {len(test)} samples ({len(test)/len(df)*100:.1f}%)')

In [ ]:
# Verify stratification
print('\nChurn rate by split:')
print(f'Train: {train["Churn_Binary"].mean()*100:.2f}%')
print(f'Validation: {val["Churn_Binary"].mean()*100:.2f}%')
print(f'Test: {test["Churn_Binary"].mean()*100:.2f}%')
print(f'Overall: {df["Churn_Binary"].mean()*100:.2f}%')

## 6. Save Processed Data

In [ ]:
# Get feature columns for modeling
feature_cols = [col for col in df.columns if 
                col.endswith('_scaled') or col.endswith('_encoded') or
                col in ['HighValue', 'ShortTermContract', 'AutomaticPayment', 'TotalServices']]

print(f'Number of features for modeling: {len(feature_cols)}')
print('\nFeature columns:')
for i, col in enumerate(feature_cols, 1):
    print(f'{i}. {col}')

In [ ]:
# Save processed full dataset
df.to_csv(PROCESSED_DATA_DIR / 'telco_churn_processed.csv', index=False)
print(f'Saved processed data to {PROCESSED_DATA_DIR / "telco_churn_processed.csv"}')

# Save train/val/test splits
train.to_csv(TRAIN_DATA_DIR / 'train.csv', index=False)
val.to_csv(VAL_DATA_DIR / 'val.csv', index=False)
test.to_csv(TEST_DATA_DIR / 'test.csv', index=False)

print(f'Saved train data to {TRAIN_DATA_DIR / "train.csv"}')
print(f'Saved validation data to {VAL_DATA_DIR / "val.csv"}')
print(f'Saved test data to {TEST_DATA_DIR / "test.csv"}')

In [ ]:
# Save transformers for inference
joblib.dump(scaler, MODELS_DIR / 'scaler.pkl')
joblib.dump(encoders, MODELS_DIR / 'label_encoder.pkl')

print(f'Saved scaler to {MODELS_DIR / "scaler.pkl"}')
print(f'Saved encoders to {MODELS_DIR / "label_encoder.pkl"}')

In [ ]:
# Summary
print('\n' + '='*60)
print('FEATURE ENGINEERING COMPLETE!')
print('='*60)
print(f'''
Summary:
- Original features: 21
- New engineered features: 7
- Encoded features: {len(encoded_cols)}
- Scaled features: {len(scaled_cols)}
- Total features for modeling: {len(feature_cols)}

Data splits:
- Train: {len(train):,} samples
- Validation: {len(val):,} samples  
- Test: {len(test):,} samples

Files saved:
- Processed data: data/processed/
- Train/Val/Test splits: data/train/, data/val/, data/test/
- Transformers: models_saved/
''')